In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import joblib
import numpy as np
import sys, os



---------------------

Add the current project folder to Python’s path and import the required custom functions for building, training, saving, and evaluating the model.


-------------------------------------------

In [3]:

sys.path.append(os.getcwd()) 

from Mymodel import (
    save_architecture_and_params,
    save_results,
    compile_model,
    train_and_save_model,
    evaluate_model,
    load_data
)


-------------------------------------------

----------------------

This script loads the saved train/validation/test splits and the label encoder so the class names are consistent during training and evaluation.

It defines several Conv1D CNN architectures (small/medium/big variants) for the non-aromatic classification task.

It then trains each architecture, saves the best checkpoint and model settings, and stores the final test results (including confusion matrix) in a separate output folder per model.

-------------------------------

------------------------

In [ ]:

# Paths
split_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits"
base_dir = os.path.abspath(".")
model_root = os.path.join(base_dir, "model_NONAROMATIC", "grouping_models_NONARPMATIC_4")

# Data
X_train, y_train, X_valid, y_valid, X_test, y_test = load_data(split_path)

# Load the label encoder and get class names
le = joblib.load(os.path.join(split_path, "label_encoder.pkl"))
label_names = list(le.classes_)
print("Class names for evaluation:", label_names)

def build_cnn_small_1(input_shape, num_classes):
    # Like cnn_small, but add another Conv1D layer
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_small_2(input_shape, num_classes):
    # Deeper version with more Conv1D layers
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium_1(input_shape, num_classes):
    # Like cnn_medium, but deeper
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium_2(input_shape, num_classes):
    # Even deeper and with dropout for regularization
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_big_2(input_shape, num_classes):
    # Even bigger model with more layers and units
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(256, 7, activation='relu'),
        tf.keras.layers.Conv1D(256, 5, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(512, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model
architectures = {
    "cnn_small_NA": build_cnn_small,
    "cnn_small_1_NA": build_cnn_small_1,
    "cnn_small_2_NA": build_cnn_small_2,
    "cnn_medium_NA": build_cnn_medium,
    "cnn_medium_1_NA": build_cnn_medium_1,
    "cnn_medium_2_NA": build_cnn_medium_2,
    "cnn_big_NA": build_cnn_big,
    "cnn_big_2_NA": build_cnn_big_2,
}


# Training config
epochs = 300
batch_size = 2048

for i, (arch_name, arch_fn) in enumerate(architectures.items(), 1):
    output_dir = os.path.join(model_root, f"{i:02d}_{arch_name}")
    os.makedirs(output_dir, exist_ok=True)
    checkpoint = ModelCheckpoint(os.path.join(output_dir, "best_model.h5"), save_best_only=True, monitor='val_accuracy', mode='max')
    callbacks = [checkpoint]

    print(f"\n{'='*10} Training architecture: {arch_name} {'='*10}")
    # Build and compile model
    model = compile_model(X_train.shape[1:], len(label_names), arch_fn)
    model_params = {"architecture_name": arch_name}

    # Save architecture and params
    save_architecture_and_params(model, model_params, output_dir)

    # Train
    history = train_and_save_model(
        model, X_train, y_train, X_valid, y_valid, output_dir,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    test_loss, test_acc = evaluate_model(model, X_test, y_test, output_dir)

    # Predict for confusion matrix and classification report
    y_pred = model.predict(X_test).argmax(axis=1)
    save_results(test_loss, test_acc, y_test, y_pred, label_names, output_dir)

    print(f"✅ Results (including confusion matrix with class names) saved in {output_dir}")


In [ ]:

split_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits"
base_dir = os.path.abspath(".")
model_root = os.path.join(base_dir, "model", "grouping_models2")

X_train, y_train, X_valid, y_valid, X_test, y_test = load_data(split_path)
le = joblib.load(os.path.join(split_path, "label_encoder.pkl"))
label_names = list(le.classes_)
print("Class names for evaluation:", label_names)
# CNN architectures
def build_cnn_small(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_big(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 7, activation='relu'),
        tf.keras.layers.Conv1D(256, 5, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

architectures = {
    "cnn_small": build_cnn_small,
    "cnn_medium": build_cnn_medium,
    "cnn_big": build_cnn_big,
}

# Training config
epochs = 200
batch_size = 2048

for i, (arch_name, arch_fn) in enumerate(architectures.items(), 1):
    output_dir = os.path.join(model_root, f"{i:02d}_{arch_name}")
    os.makedirs(output_dir, exist_ok=True)
    checkpoint = ModelCheckpoint(os.path.join(output_dir, "best_model.h5"), save_best_only=True, monitor='val_accuracy', mode='max')
    callbacks = [checkpoint]

    print(f"\n{'='*10} Training architecture: {arch_name} {'='*10}")
    # Build and compile model
    model = compile_model(X_train.shape[1:], len(label_names), arch_fn)
    model_params = {"architecture_name": arch_name}

    # Save architecture and params
    save_architecture_and_params(model, model_params, output_dir)

    # Train
    history = train_and_save_model(
        model, X_train, y_train, X_valid, y_valid, output_dir,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    test_loss, test_acc = evaluate_model(model, X_test, y_test, output_dir)

    y_pred = model.predict(X_test).argmax(axis=1)
    save_results(test_loss, test_acc, y_test, y_pred, label_names, output_dir)

    print(f"✅ Results (including confusion matrix with class names) saved in {output_dir}")


Class names for evaluation: ['_AROMATIC_OTHER', '_CHN_AROMATIC', '_CHONS_AROMATIC', '_CHON_AROMATIC', '_CHO_AROMATIC', '_CH_AROMATIC']

========== Training architecture: cnn_small ==========
Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 596ms/step - accuracy: 0.2023 - loss: 155.5695

46/46 ━━━━━━━━━━━━━━━━━━━━ 30s 614ms/step - accuracy: 0.2033 - loss: 154.1514 - val_accuracy: 0.3505 - val_loss: 14.4398
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 583ms/step - accuracy: 0.2899 - loss: 11.4002 - val_accuracy: 0.3218 - val_loss: 6.3205
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 573ms/step - accuracy: 0.3357 - loss: 5.6434

46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 587ms/step - accuracy: 0.3359 - loss: 5.6312 - val_accuracy: 0.3695 - val_loss: 4.1314
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 584ms/step - accuracy: 0.3545 - loss: 3.8522 - val_accuracy: 0.3603 - val_loss: 3.2304
Epoch 5/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 570ms/step - accuracy: 0.3694 - loss: 3.0795

46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 584ms/step - accuracy: 0.3694 - loss: 3.0776 - val_accuracy: 0.3849 - val_loss: 2.8214
Epoch 6/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 584ms/step - accuracy: 0.3679 - loss: 2.7209 - val_accuracy: 0.3526 - val_loss: 2.5512
Epoch 7/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 581ms/step - accuracy: 0.3752 - loss: 2.4635 - val_accuracy: 0.3515 - val_loss: 2.3286
Epoch 8/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 571ms/step - accuracy: 0.3754 - loss: 2.2919

46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 584ms/step - accuracy: 0.3754 - loss: 2.2910 - val_accuracy: 0.4154 - val_loss: 2.1909
Epoch 9/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 584ms/step - accuracy: 0.3811 - loss: 2.1364 - val_accuracy: 0.3797 - val_loss: 2.0305
Epoch 10/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 27s 593ms/step - accuracy: 0.3807 - loss: 1.9985 - val_accuracy: 0.3823 - val_loss: 1.9086
Epoch 11/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 586ms/step - accuracy: 0.3807 - loss: 1.9070

...

46/46 ━━━━━━━━━━━━━━━━━━━━ 201s 4s/step - accuracy: 0.7367 - loss: 0.6973 - val_accuracy: 0.7226 - val_loss: 0.7406
Epoch 95/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7339 - loss: 0.7057 - val_accuracy: 0.6992 - val_loss: 0.7799
Epoch 96/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 200s 4s/step - accuracy: 0.7339 - loss: 0.7050 - val_accuracy: 0.7123 - val_loss: 0.7572
Epoch 97/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7377 - loss: 0.6962 - val_accuracy: 0.7210 - val_loss: 0.7355
Epoch 98/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 201s 4s/step - accuracy: 0.7411 - loss: 0.6890 - val_accuracy: 0.7128 - val_loss: 0.7445
Epoch 99/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 203s 4s/step - accuracy: 0.7443 - loss: 0.6818 - val_accuracy: 0.7178 - val_loss: 0.7482
Epoch 100/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 203s 4s/step - accuracy: 0.7486 - loss: 0.6723 - val_accuracy: 0.7097 - val_loss: 0.7610
Epoch 101/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7419 - loss: 0.6844

46/46 ━━━━━━━━━━━━━━━━━━━━ 201s 4s/step - accuracy: 0.7419 - loss: 0.6844 - val_accuracy: 0.7252 - val_loss: 0.7293
Epoch 102/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7492 - loss: 0.6685 - val_accuracy: 0.7194 - val_loss: 0.7548
Epoch 103/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 276s 5s/step - accuracy: 0.7450 - loss: 0.6786 - val_accuracy: 0.7100 - val_loss: 0.7480
Epoch 104/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 208s 5s/step - accuracy: 0.7441 - loss: 0.6765 - val_accuracy: 0.7169 - val_loss: 0.7430
Epoch 105/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7433 - loss: 0.6823

46/46 ━━━━━━━━━━━━━━━━━━━━ 207s 4s/step - accuracy: 0.7433 - loss: 0.6823 - val_accuracy: 0.7295 - val_loss: 0.7200
Epoch 106/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 203s 4s/step - accuracy: 0.7529 - loss: 0.6601 - val_accuracy: 0.7169 - val_loss: 0.7515
Epoch 107/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 203s 4s/step - accuracy: 0.7516 - loss: 0.6628 - val_accuracy: 0.7088 - val_loss: 0.7510
Epoch 108/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7548 - loss: 0.6549 - val_accuracy: 0.7084 - val_loss: 0.7641
Epoch 109/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7499 - loss: 0.6673 - val_accuracy: 0.7160 - val_loss: 0.7496
Epoch 110/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 200s 4s/step - accuracy: 0.7466 - loss: 0.6747 - val_accuracy: 0.7294 - val_loss: 0.7352
Epoch 111/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7537 - loss: 0.6575

46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7537 - loss: 0.6575 - val_accuracy: 0.7327 - val_loss: 0.7150
Epoch 112/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 201s 4s/step - accuracy: 0.7526 - loss: 0.6577 - val_accuracy: 0.7239 - val_loss: 0.7311
Epoch 113/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 204s 4s/step - accuracy: 0.7552 - loss: 0.6557 - val_accuracy: 0.7326 - val_loss: 0.7205
Epoch 114/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 203s 4s/step - accuracy: 0.7560 - loss: 0.6488 - val_accuracy: 0.7262 - val_loss: 0.7407
Epoch 115/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.7481 - loss: 0.6672 - val_accuracy: 0.7222 - val_loss: 0.7590
Epoch 116/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 203s 4s/step - accuracy: 0.7589 - loss: 0.6437 - val_accuracy: 0.7222 - val_loss: 0.7469
Epoch 117/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 201s 4s/step - accuracy: 0.7579 - loss: 0.6431 - val_accuracy: 0.7262 - val_loss: 0.7121
Epoch 118/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 201s 4s/step - accuracy: 0.7589 - loss: 0.6345 - val_accuracy: 

✅ Model evaluation saved to c:\Users\moham\Desktop\Thesis start\code from lazar\Multitask_Classifier_with_grouping_classification\model\grouping_models2\02_cnn_medium\evaluation.txt
364/364 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step
Saved confusion matrix with class names:
                 _AROMATIC_OTHER  _CHN_AROMATIC  _CHONS_AROMATIC  \
_AROMATIC_OTHER             3018              5              257   
_CHN_AROMATIC                 29            166                3   
_CHONS_AROMATIC              173              6             1146   
_CHON_AROMATIC               252             81               83   
_CHO_AROMATIC                 60             13               17   
_CH_AROMATIC                   4              3                0   

                 _CHON_AROMATIC  _CHO_AROMATIC  _CH_AROMATIC  
_AROMATIC_OTHER             495            259             1  
_CHN_AROMATIC               357             46             3  
_CHONS_AROMATIC             243             63             2  
_CHON_

-------------------------------------------

This script loads the saved dataset splits and label encoder, then defines deeper CNN architectures with more layers and parameters to increase model capacity for the non-aromatic classification task.
Each architecture is trained for 300 epochs, with the best validation model checkpoint saved automatically.
Finally, every model is evaluated on the test set, and the results (including confusion matrix and metrics) are stored in separate output folders for comparison.

-------------------------------------------

In [ ]:

split_path = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits"
base_dir = os.path.abspath(".")
model_root = os.path.join(base_dir, "model_NA", "grouping_models_NA2")

# Data
X_train, y_train, X_valid, y_valid, X_test, y_test = load_data(split_path)

# Load the label encoder and get class names
le = joblib.load(os.path.join(split_path, "label_encoder.pkl"))
label_names = list(le.classes_)
print("Class names for evaluation:", label_names)

def build_cnn_small_1(input_shape, num_classes):
    # Like cnn_small, but add another Conv1D layer
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_small_2(input_shape, num_classes):
    # Deeper version with more Conv1D layers
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium_1(input_shape, num_classes):
    # Like cnn_medium, but deeper
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_medium_2(input_shape, num_classes):
    # Even deeper and with dropout for regularization
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(128, 5, activation='relu'),
        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn_big_2(input_shape, num_classes):
    # Even bigger model with more layers and units
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(256, 7, activation='relu'),
        tf.keras.layers.Conv1D(256, 5, activation='relu'),
        tf.keras.layers.Conv1D(256, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(512, 3, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


architectures = {
    "cnn_small_NA": build_cnn_small_1,
    "cnn_small_1_NA": build_cnn_small_1,
    "cnn_small_2_NA": build_cnn_small_2,
    "cnn_medium_NA": build_cnn_medium_1,
    "cnn_medium_1_NA": build_cnn_medium_1,
    "cnn_medium_2_NA": build_cnn_medium_2,
    "cnn_big_NA": build_cnn_big,
    "cnn_big_2_NA": build_cnn_big_2,
}


# Training config
epochs = 300
batch_size = 2048

for i, (arch_name, arch_fn) in enumerate(architectures.items(), 1):
    output_dir = os.path.join(model_root, f"{i:02d}_{arch_name}")
    os.makedirs(output_dir, exist_ok=True)
    checkpoint = ModelCheckpoint(os.path.join(output_dir, "best_model.h5"), save_best_only=True, monitor='val_accuracy', mode='max')
    callbacks = [checkpoint]

    print(f"\n{'='*10} Training architecture: {arch_name} {'='*10}")
    # Build and compile model
    model = compile_model(X_train.shape[1:], len(label_names), arch_fn)
    model_params = {"architecture_name": arch_name}

    # Save architecture and params
    save_architecture_and_params(model, model_params, output_dir)

    # Train
    history = train_and_save_model(
        model, X_train, y_train, X_valid, y_valid, output_dir,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    test_loss, test_acc = evaluate_model(model, X_test, y_test, output_dir)

    # Predict for confusion matrix and classification report
    y_pred = model.predict(X_test).argmax(axis=1)
    save_results(test_loss, test_acc, y_test, y_pred, label_names, output_dir)

    print(f"✅ Results (including confusion matrix with class names) saved in {output_dir}")


Class names for evaluation: ['_CHN_NON_AROMATIC', '_CHONS_NON_AROMATIC', '_CHON_NON_AROMATIC', '_CHO_NON_AROMATIC', '_CH_NON_AROMATIC', '_NON_AROMATIC_OTHER']

========== Training architecture: cnn_small_NA ==========
Epoch 1/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2468 - loss: 66.4168

26/26 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.2472 - loss: 65.2862 - val_accuracy: 0.3243 - val_loss: 8.2425
Epoch 2/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3028 - loss: 7.2483

26/26 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.3029 - loss: 7.2031 - val_accuracy: 0.3410 - val_loss: 3.7454
Epoch 3/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3466 - loss: 3.5024

26/26 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.3473 - loss: 3.4923 - val_accuracy: 0.3970 - val_loss: 2.6861
Epoch 4/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3965 - loss: 2.5633

26/26 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.3965 - loss: 2.5598 - val_accuracy: 0.4005 - val_loss: 2.2748
Epoch 5/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4016 - loss: 2.2025

...

26/26 ━━━━━━━━━━━━━━━━━━━━ 1698s 65s/step - accuracy: 0.7154 - loss: 0.7898 - val_accuracy: 0.7459 - val_loss: 0.7098
Epoch 13/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 63s/step - accuracy: 0.7274 - loss: 0.7554  

26/26 ━━━━━━━━━━━━━━━━━━━━ 1696s 65s/step - accuracy: 0.7271 - loss: 0.7558 - val_accuracy: 0.7503 - val_loss: 0.6950
Epoch 14/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 60s/step - accuracy: 0.7381 - loss: 0.7272  

26/26 ━━━━━━━━━━━━━━━━━━━━ 1613s 62s/step - accuracy: 0.7381 - loss: 0.7269 - val_accuracy: 0.7708 - val_loss: 0.6544
Epoch 15/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 61s/step - accuracy: 0.7457 - loss: 0.7016  

26/26 ━━━━━━━━━━━━━━━━━━━━ 1642s 63s/step - accuracy: 0.7458 - loss: 0.7015 - val_accuracy: 0.7732 - val_loss: 0.6463
Epoch 16/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 58s/step - accuracy: 0.7536 - loss: 0.6789 

26/26 ━━━━━━━━━━━━━━━━━━━━ 1546s 59s/step - accuracy: 0.7538 - loss: 0.6787 - val_accuracy: 0.7897 - val_loss: 0.6017
Epoch 17/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 1562s 60s/step - accuracy: 0.7718 - loss: 0.6392 - val_accuracy: 0.7817 - val_loss: 0.6170
Epoch 18/300
16/26 ━━━━━━━━━━━━━━━━━━━━ 9:28 57s/step - accuracy: 0.7763 - loss: 0.6243 